In [ ]:
# R2_CAR171_Canopus.ipynb
# Estimating the stray-light from Canopus star for the Commissioning Activity Request (CAR) 171. 
# This notebook is based on the R1_Rosalia_stray_example.ipynb notebook, but it is adapted to the specific case of Canopus.
# - Alejandro S. Borlaff / NASA Ames / a.s.borlaff@nasa.gov. March 11, 2026.
import os
import rosalia as rs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.time import Time
print("ROSALIA version:", rs.__version__)
plt.style.use(os.path.dirname(rs.__file__) + "/style/nature_style.mplstyle")

In [ ]:
# Let's find Canopus (alpha Carinae), the star selected for CAR171. 
# This star is in the continuous viewing zone of Roman, and it is the second brightest star in the sky.
import rosalia as rs
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord
from astroquery.simbad import Simbad

alfCar = Simbad.query_object('Canopus')
ra_star = alfCar["ra"][0]
dec_star = alfCar["dec"][0]

target = SkyCoord(ra_star*u.deg, dec_star*u.deg, frame="icrs")

date = Time('2026-07-01T00:00:00.0', format='isot', scale='utc')

offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=ra_star,
                                                      dec_target=dec_star,
                                                      mjd=date.mjd,
                                                      dX=0, dY=0)

print("Offset pointing for Canopus:", offset_pointing)

In [ ]:
# Equatorial pole Test
from astropy.coordinates import get_body
date = Time('2026-07-01T00:00:00.0', format='isot', scale='utc')

sun_coord = get_body('Sun', date)   # get coordinate object for the Sun for each day of the year

ecliptic_pole = SkyCoord(0*u.deg, 89*u.deg, frame="barycentricmeanecliptic")
ecliptic_pole = ecliptic_pole.transform_to("icrs")
bestPA_for_canopus_Dec26 = rs.telescopes.Roman.get_bestPA(ra=ra_star, dec=dec_star, mjd=date.mjd)
print(bestPA_for_canopus_Dec26)
offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=ecliptic_pole.ra.degree,
                                                      dec_target=ecliptic_pole.dec.degree,
                                                      mjd=date.mjd,
                                                      dX=0, dY=0)
print(offset_pointing)
print(sun_coord.barycentricmeanecliptic)

In [ ]:
print(rs.telescopes.Roman.get_bestPA(ra=ra_star, dec=dec_star, mjd=date.mjd))


In [ ]:
sun_ra = sun_coord.ra.degree
sun_dec = sun_coord.dec.degree
plt.scatter(sun_ra, sun_dec, marker="s", s=100)
plt.scatter(ra_star, dec_star, marker="o")
plt.xlim((0,360))
plt.ylim((-90,90))

In [ ]:
CAR171 = pd.read_csv("CAR171/CAR171_v2_coordinates.csv")
CAR171

In [ ]:
# These are the coordinates of specially sensitive locations 
# for stray-light, measured in offset degrees from the center of WFI focal plane
dX = CAR171["MPA_ThetaX"]
dY = CAR171["MPA_ThetaY"]
CAR171["RA_Source"] = ra_star
CAR171["Dec_Source"] = dec_star
ra_source = CAR171["RA_Source"]
dec_source = CAR171["Dec_Source"]
number_of_points_of_interest = len(dX)

# To measure the stray-light from those sources, we need to place the center 
# of Roman / WFI at a certain distance and position angle from the source 
# that generates the stray-light. We will name that source the "offending" source. 

# ROSALIA has a specific tool to compute those locations. 

# The optimal position angle of Roman depends with time. 
# Let's set an approximate time for the Commissioning Activities
date = Time('2026-12-01T00:00:00.0', format='isot', scale='utc')

ra_wfi = np.zeros(number_of_points_of_interest)
dec_wfi = np.zeros(number_of_points_of_interest)
V3PA_ori = np.zeros(number_of_points_of_interest)
V3PA_off = np.zeros(number_of_points_of_interest)
WFIPA_off = np.zeros(number_of_points_of_interest)

activity_name_apt = []
category = []
description = []
for i in range(number_of_points_of_interest):
    offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=ra_source[i],
                                                      dec_target=dec_source[i],
                                                      mjd=date.mjd,
                                                      dX=dX[i], dY=dY[i])
    ra_wfi[i] = offset_pointing["ra_wficen"]
    dec_wfi[i] = offset_pointing["dec_wficen"]
    WFIPA_off[i] = offset_pointing["PA_WFI_offset"]%360
    V3PA_off[i] = offset_pointing["V3PA_offset"]%360
    V3PA_ori[i] = offset_pointing["V3PA_origin"]%360
    # v3pa[i] = offset_pointing["PA_v3"] % 360
    activity_name_apt.append(CAR171["Activity name"].iloc[i]+"_"+str(i+1).zfill(3))
    category.append("Calibration")
    description.append("Stray light test")

car171_APT = pd.DataFrame({"Target": activity_name_apt, "ArchiveTarget": activity_name_apt, "description":description, "category": category, "RA": ra_wfi, "DEC": dec_wfi, 
                           "WFIPA_off": WFIPA_off, "V3PA_off": V3PA_off, "V3PA_ori": V3PA_ori})
car171_APT.to_csv("CAR171_apt_targets.csv")


In [ ]:
import pandas as pd
canopus_catalog = pd.DataFrame({"ra": ra_star, 
                             "dec": dec_star, 
                             "source_id": "Canopus", 
                             "cat_id": 1,
                             "mag_lambda": -0.74}, index=[0])

# if True:
for i in range(len(ra_wfi)):
    #i = 0
    ra = ra_wfi[i]
    dec = dec_wfi[i]
    pa = WFIPA_off[i]
    mjd = date.mjd
    bandpass="F129"
    exptime=600
    rosalia_stray = rs.correct.rosalia_stray(ra=ra, dec=dec, prefix="CAR171_" + CAR171["Activity name"].iloc[i] + "_",
                                             PA=pa, date=date, bandpass=bandpass, 
                                             exptime=exptime, radius=1, catalog=canopus_catalog,
                                             g_mag_max=17)

In [ ]:
for i in range(number_of_points_of_interest):
    ra = ra_wfi[i]
    dec = dec_wfi[i]
    pa = pa_wfi[i]
    mjd = date.mjd
    bandpass="F129"
    exptime=600
    rosalia_stray = rs.correct.rosalia_stray(ra=ra, dec=dec, 
                                             PA=pa, date=date, bandpass=bandpass, 
                                             exptime=exptime, radius=1,
                                             g_mag_max=15, figsize=(7,6), mu_vmin=22, mu_vmax=27)

In [ ]:
# For the Dragon's Breath example, let's change the target to a dimmer star. 
# Magnitude 2 star. 

# Let's find Canopus (alpha Carinae), the star selected for CAR171. 
# This star is in the continuous viewing zone of Roman, and it is the second brightest star in the sky.
import astropy.units as u
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord

alfUrsae = Simbad.query_object('Mizar')
ra_star = alfUrsae["ra"][0]
dec_star = alfUrsae["dec"][0]

target = SkyCoord(ra_star*u.deg, dec_star*u.deg, frame="icrs")
print(target)

# These are the coordinates of specially sensitive locations 
# for stray-light, measured in offset degrees from the center of WFI focal plane
dX = np.array([-0.0688, 0.0688, -0.138, 0.138])
dY = np.array([-0.0312-20/60/60, -0.0312, -0.0035, -0.0048])
number_of_points_of_interest = len(dX)


# To measure the stray-light from those sources, we need to place the center 
# of Roman / WFI at a certain distance and position angle from the source 
# that generates the stray-light. We will name that source the "offending" source. 

# ROSALIA has a specific tool to compute those locations. 

# The optimal position angle of Roman depends with time. 
# Let's set an approximate time for the Commissioning Activities
from astropy.time import Time
date = Time('2026-11-21T00:00:00.0', format='isot', scale='utc')

ra_wfi = np.zeros(number_of_points_of_interest)
dec_wfi = np.zeros(number_of_points_of_interest)
pa_wfi = np.zeros(number_of_points_of_interest)

for i in range(number_of_points_of_interest):
    offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=target.ra.degree,
                                                      dec_target=target.dec.degree,
                                                      mjd=date.mjd,
                                                      dX=dX[i], dY=dY[i])
    ra_wfi[i] = offset_pointing["ra_wficen"]
    dec_wfi[i] = offset_pointing["dec_wficen"]
    pa_wfi[i] = offset_pointing["pa_wfi"]


In [ ]:
rs.detectors.fe2mu(1, instrument = "WFI", filter_name="F129", telescope="Roman")

In [ ]:
fe2mu_png = rs.plots.make_stray_plot(input_name="/Users/aborlaff/NASA/ROSALIA/notebooks/Dragon_negative_WFI_F129_RA_200.985_DEC_055.003_MJD_61365.00000_PA_063.57_stray_drz_scaled.fits",
                                         ext=1, mode="fe2mu",
                                         color_label = "Surface brightness (mag arcsec$^{-2}$)", 
                                         figsize=(7,6),
                                         mu_vmin=28, 
                                         mu_vmax=22)

stars_around_png = rs.plots.make_stars_around_plot(flt_name="/Users/aborlaff/NASA/ROSALIA/notebooks/Dragon_negative_WFI_F129_RA_200.985_DEC_055.003_MJD_61365.00000_PA_063.57.fits", 
                                                   catalog_name="/Users/aborlaff/NASA/ROSALIA/notebooks/Dragon_negative_WFI_F129_RA_200.985_DEC_055.003_MJD_61365.00000_PA_063.57_source_catalog.csv", 
                                                   output_name=None, figsize=(7,6))


In [ ]:
fe2mu_png = rs.plots.make_stray_plot(input_name="/Users/aborlaff/NASA/ROSALIA/notebooks/Dragon_WFI_F129_RA_200.977_DEC_055.001_MJD_61365.00000_PA_063.57_stray_drz_scaled.fits",
                                         ext=1, mode="fe2mu",
                                         color_label = "Surface brightness (mag arcsec$^{-2}$)", 
                                         figsize=(7,6),
                                         mu_vmin=28, 
                                         mu_vmax=22)


stars_around_png = rs.plots.make_stars_around_plot(flt_name="/Users/aborlaff/NASA/ROSALIA/notebooks/Dragon_WFI_F129_RA_200.977_DEC_055.001_MJD_61365.00000_PA_063.57.fits", 
                                                   catalog_name="/Users/aborlaff/NASA/ROSALIA/notebooks/Dragon_negative_WFI_F129_RA_200.985_DEC_055.003_MJD_61365.00000_PA_063.57_source_catalog.csv", 
                                                   output_name=None, figsize=(7,6))

In [ ]:
# NEGATIVE TARGET 

from astropy.time import Time
date = Time('2026-11-21T00:00:00.0', format='isot', scale='utc')

number_of_points_of_interest = 2
dX_negative = np.array([-0.06651558333333384, 0.06651558333333384]) 
dY_negative = np.array([-0.17538308333333466, -0.17538308333333466])

ra_wfi = np.zeros(number_of_points_of_interest)
dec_wfi = np.zeros(number_of_points_of_interest)
pa_wfi = np.zeros(number_of_points_of_interest)


alfCar = Simbad.query_object('Canopus')
ra_star = alfCar["ra"][0]
dec_star = alfCar["dec"][0]

import pandas as pd
canopus_catalog = pd.DataFrame({"ra": ra_star, 
                             "dec": dec_star, 
                             "source_id": "Canopus", 
                             "cat_id": 1,
                             "mag_lambda": -0.74}, index=[0])


for i in range(number_of_points_of_interest):
    offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=target.ra.degree,
                                                      dec_target=target.dec.degree,
                                                      mjd=date.mjd,
                                                      dX=dX_negative[i], dY=dY_negative[i])
    ra_wfi[i] = offset_pointing["ra_wficen"]
    dec_wfi[i] = offset_pointing["dec_wficen"]
    pa_wfi[i] = offset_pointing["pa_wfi"]


if True:
    i = 1
    ra = ra_wfi[i]
    dec = dec_wfi[i]
    pa = pa_wfi[i]
    mjd = date.mjd
    bandpass="F129"
    exptime=600
    rosalia_stray = rs.correct.rosalia_stray(ra=ra, dec=dec, prefix="Dragon_negative_",
                                             PA=pa, date=date, bandpass=bandpass, 
                                             exptime=exptime, radius=1, catalog=canopus_catalog,
                                             g_mag_max=17)

In [ ]:
texp = (3*60+16)*4 + 200 + 10 
print(texp/60)

In [ ]:
85/6*7

In [ ]:
(200+100+100+382.5+28+184+100+28)/60*(1.15)